## NoteBook to explore the conversion between the gathered data catelouge into a time series data frame for Model Training and Testing

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

### PGA Look Up Class

In [25]:
class PGALookUp:
	"""A class to look up PGA values based on MMI levels."""
	_pga_look_up = {
		2.0: 0.002268553,
		2.5: 0.00312874,
		3.0: 0.0055180005,
		3.5: 0.010397652,
		4.0: 0.018992,
		4.5: 0.027748000000000002,
		5.0: 0.034651877,
		5.5: 0.058811,
		6.0: 0.069783,
		6.5: 0.10616,
		7.0: 0.13697,
		7.5: 0.20955,
		8.0: 0.27436
	}

	_lowest_mmi = min(_pga_look_up.keys())
	_highest_mmi = max(_pga_look_up.keys())
	@staticmethod
	def get_pga(mmi:float) -> float:
		"""Returns the median PGA value (in g) corresponding to a given MMI level.

		Note:
			This conversion function is strictly a reference utility and is not used
			within the training or evaluation/test pipelines.
		"""
		
		lookup = PGALookUp()._pga_look_up
		if mmi in lookup:
			return lookup[mmi]
		else:
			raise ValueError(f"MMI value {mmi} not found in lookup table.")
print(PGALookUp.get_pga(5.0))  # Example usage

0.034651877


## Radius Model Coeffifients

In [32]:
# formula, log_r_hypo ~ a + b * log_pga + c * magnitude + d * log_vs_30 + e * log_depth

def get_r_epi(
	pga:float,
	magnitude: float,
	vs_30: float,
	depth: float
) -> float:
	"""
	Estimate epicentral distance (r_epi) from earthquake magnitude,
	site conditions, focal depth, and a target Modified Mercalli
	Intensity (MMI).

	The function first converts the supplied MMI value to Peak Ground
	Acceleration (PGA) using ``pga_func``. PGA, magnitude, VS30, and
	depth are then passed through an empirical regression model trained
	on USGS intensity records from active tectonic regions.

	Regression model:

		log10(r_hypo) =
			a
			+ b * log10(PGA)
			+ c * magnitude
			+ d * log10(VS30 / 760)
			+ e * log10(depth)

	where:

		r_hypo : hypocentral distance (km)
		PGA    : peak ground acceleration (g)
		VS30   : average shear-wave velocity in the upper 30 m (m/s)

	After estimating hypocentral distance, epicentral distance is
	computed using:

		r_epi = sqrt(r_hypo² - depth²)

	Parameters
	----------
	pga : float
		Target Peak Ground Acceleration in units of g.

	magnitude : float
		Earthquake magnitude.

	vs_30 : float
		Average shear-wave velocity in the upper 30 meters (m/s).

	depth : float
		Earthquake focal depth (km).

	Returns
	-------
	float
		Estimated epicentral distance (km). Returns 0.0 if the computed
		hypocentral distance is smaller than the focal depth.

	Notes
	-----
	The regression coefficients were derived from USGS observations in
	active tectonic regions and are intended for use in Nepal and other
	tectonically active environments.
	"""
	
	log_pga = np.log10(pga)
	log_vs_30 = np.log10(vs_30/760)
	log_depth = np.log10(depth)
	a=b=c=d=e=0
	if magnitude <= 5.0:  # mse = 0.041502, mae = 0.155973
		a = -1.0077478524404035
		b = -0.32296656 
		c = 0.43818554 
		d = 0.04913303 
		e = 0.07811176
	else:    # mse = 0.022438, mae = 0.112091
		a = -0.473103270179728
		b = -0.37554408 
		c = 0.28561204
		d = -0.09830627
		e = 0.0714735

	rhypo = 10**(a + b * log_pga + c * magnitude + d * log_vs_30 + e * log_depth ) 
	sqr_r_epi = rhypo**2 - depth**2
	r_epi = np.sqrt(sqr_r_epi) if sqr_r_epi > 0.0 else 0.0
	return r_epi

print("Using Median PGA Lookup:")
print(get_r_epi(pga=PGALookUp.get_pga(2.5), magnitude=7.8, vs_30=270, depth=8.0))  # Example usage


Using Median PGA Lookup:
636.7460688235639
